---
title: "Financial Statistics -  Coursework"
author: Group 30
date: "23 November 2025"
institute: "Imperial Business School"
format:
  pdf:
    fig-cap-location: top
    include-in-header:
      text: |
        \usepackage[labelfont=bf,textfont=bf]{caption}   
execute:
  warning: false
  message: false
---
\newpage
\tableofcontents
\listoffigures
\listoftables
\newpage

In [15]:
# Load libraries
library(readxl)
library(lubridate)
library(readr)


# Question 1 - CAPM

## 1. Prepare the Dataset

In [16]:
# Use the dataset data coursework Q1
data_q1 <- read_excel("data_coursework_Q1.xls")


data_q1$date <- make_date(year = data_q1$year_, month = data_q1$month_, day = 1)
data_q1$IBM <- parse_number(data_q1$IBM)
df_q1 <- data_q1[, c("date","SP500", "IBM", "1-month Tbill")]

colnames(df_q1)[colnames(df_q1) == "SP500"] <- "Rm"
colnames(df_q1)[colnames(df_q1) == "1-month Tbill"] <- "Rf"
colnames(df_q1)[colnames(df_q1) == "IBM"] <- "Ri"

head(df_q1)

New names:
• `` -> `...5`
• `` -> `...8`
• `` -> `...9`
• `` -> `...10`
• `` -> `...11`
• `` -> `...13`
• `` -> `...14`
Warning message:
“3 parsing failures.
row col expected   actual
  1  -- a number adjusted
  2  -- a number closed  
  3  -- a number pirce   
”


date,Rm,Ri,Rf
<date>,<dbl>,<dbl>,<dbl>
1960-01-01,58.03,NA,0.33
1960-02-01,55.78,NA,0.29
1960-03-01,55.02,NA,0.35
1960-04-01,55.73,NA,0.19
1960-05-01,55.22,NA,0.27
1960-06-01,57.26,NA,0.24


In [17]:
# Construct the measures

# excess return for IBM (Ri - Rf)
df_q1$ri_rf <- df_q1$Ri - df_q1$Rf
# excess market return (Rm - Rf)
df_q1$rm_rf <- df_q1$Rm - df_q1$Rf
# squared market excess return
df_q1$rm_rf_sq <- df_q1$rm_rf^2

# D_t = 1 if market excess return > 0, 0 otherwise
df_q1$D <- ifelse(df_q1$rm_rf > 0, 1, 0)
# (1 - D_t)
df_q1$D_inv <- 1 - df_q1$D

# interaction terms
df_q1$D_rm    <- df_q1$D * df_q1$rm_rf
df_q1$Dinv_rm <- df_q1$D_inv * df_q1$rm_rf

## 2. Estimate linear regression model

In [18]:
# Model 1: Standard CAPM
# ri_rf = alpha + beta * rm_rf + u
capm_model <- lm(ri_rf ~ rm_rf, data = df_q1)
# show summary
summary(capm_model)

# - Model 2: Extended CAPM
# ri_rf = alpha + beta1*D_rm + beta2*Dinv_rm + beta3*rm_rf_sq + u
ext_model <- lm(ri_rf ~ D_rm + Dinv_rm + rm_rf_sq, data = df_q1)
# show summary
summary(ext_model)


Call:
lm(formula = ri_rf ~ rm_rf, data = df_q1)

Residuals:
     Min       1Q   Median       3Q      Max 
-10.0393  -1.7344  -0.1952   0.7346   9.3482 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) -1.735575   0.332108  -5.226    3e-07 ***
rm_rf        0.083115   0.002128  39.058   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.034 on 346 degrees of freedom
  (24 observations deleted due to missingness)
Multiple R-squared:  0.8151,	Adjusted R-squared:  0.8146 
F-statistic:  1526 on 1 and 346 DF,  p-value: < 2.2e-16



Call:
lm(formula = ri_rf ~ D_rm + Dinv_rm + rm_rf_sq, data = df_q1)

Residuals:
    Min      1Q  Median      3Q     Max 
-5.0503 -1.1971 -0.1176  1.2738  7.0692 

Coefficients: (1 not defined because of singularities)
              Estimate Std. Error t value Pr(>|t|)    
(Intercept) -1.458e+01  6.102e-01  -23.89   <2e-16 ***
D_rm         2.585e-01  7.929e-03   32.60   <2e-16 ***
Dinv_rm             NA         NA      NA       NA    
rm_rf_sq    -4.524e-04  2.015e-05  -22.45   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 1.937 on 345 degrees of freedom
  (24 observations deleted due to missingness)
Multiple R-squared:  0.9249,	Adjusted R-squared:  0.9244 
F-statistic:  2124 on 2 and 345 DF,  p-value: < 2.2e-16


## 3. F-test for H0: β1 = β2

In [19]:
# Restricted model: force beta1 = beta2
# ri_rf = alpha + beta * rm_rf + beta3*rm_rf_sq + u
restricted_model <- lm(ri_rf ~ rm_rf + rm_rf_sq, data = df_q1)

# F-test between restricted and unrestricted models
anova(restricted_model, ext_model)

,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,345,1294.161,NA,NA,NA,NA
2,345,1294.161,0,0,NA,NA


## 4. t-test for H0: α = 0 

In [20]:
# extract summary table
capm_summary <- summary(capm_model)$coefficients

# alpha estimate, t-value, p-value
alpha_est <- capm_summary["(Intercept)", "Estimate"]
alpha_t   <- capm_summary["(Intercept)", "t value"]
alpha_p   <- capm_summary["(Intercept)", "Pr(>|t|)"]

alpha_est
alpha_t
alpha_p

[1] -1.735575

[1] -5.225931

[1] 3.002077e-07

# Question 2 - probability of a positive asset return

# Question 3 - CIR model for the term structure of interest rate